# Bazaario Analytics — Notebook 2: Sentiment Analysis
## IS215 Digital Business — Group 4

### What this notebook does

Customer reviews like *"Amazing satay!"* or *"Overpriced and cold food"* contain valuable information, but reading hundreds of them manually is impractical. **Sentiment analysis** uses AI to automatically classify each review as positive or negative.

| Part | Approach | How it works | Speed |
|------|----------|-------------|-------|
| **A** | Hugging Face (distilbert) | A pre-trained AI model that already understands English sentiment | Fast — seconds for 800 reviews |
| **B** | Ollama LLM (llama3) | A large language model that reads each review and decides | Slow — minutes for 50 reviews |

### Why this matters for Bazaario

- **Vendors** see whether customers are happy with their food and service
- **Admins** can rank all vendors by customer satisfaction and flag underperformers
- The results feed directly into the **Vendor Analytics → Reviews tab** and **Admin Dashboard → Reviews tab** in the web app

---
# Part A: Sentiment Analysis with Hugging Face

Hugging Face hosts thousands of pre-trained AI models. We use `distilbert` — a model that was trained on millions of movie and product reviews, so it already knows what positive and negative language looks like. We just feed it our pasar malam reviews and it classifies each one.

### Step 1: Import tools

In [ ]:
import io, time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

# Hugging Face — the library that gives us access to pre-trained AI models
from transformers import pipeline

### Step 2: Load and explore the review data

Our dataset contains **800 customer reviews** from Bazaario vendors. Each review has:
- `content` — the actual review text (e.g., *"Best satay I've ever had!"*)
- `rating` — a 1-5 star rating given by the customer
- `sentiment` — our ground truth label: POSITIVE (rating 4-5) or NEGATIVE (rating 1-2)

We'll use the AI model to predict sentiment from the text alone, then compare its predictions to the actual ratings to see how accurate it is.

In [ ]:
# Load review data
df = pd.read_csv('data/reviews.csv')

print(f"Total reviews: {len(df)}")
print(f"Columns: {list(df.columns)}")
df.head()

In [ ]:
# What does the rating distribution look like?
# A healthy platform should have more positive than negative reviews
df['rating'].hist(bins=[0.5, 1.5, 2.5, 3.5, 4.5, 5.5], edgecolor='white')
plt.title('Distribution of Customer Ratings')
plt.xlabel('Star Rating')
plt.ylabel('Number of Reviews')
plt.xticks([1, 2, 3, 4, 5])
plt.show()

print("\nSentiment breakdown:")
print(df['sentiment'].value_counts())

### Step 3: Verify the data

Our ground truth mapping:
- Rating 4-5 → POSITIVE
- Rating 1-2 → NEGATIVE

The AI model will **only see the text** — not the rating. It has to figure out the sentiment from the words alone.

In [ ]:
# Show some examples of positive and negative reviews
print("=== Sample POSITIVE reviews ===")
for text in df[df['sentiment'] == 'POSITIVE']['content'].head(3):
    print(f"  ✓ {text}")

print("\n=== Sample NEGATIVE reviews ===")
for text in df[df['sentiment'] == 'NEGATIVE']['content'].head(3):
    print(f"  ✗ {text}")

# Extract just the text for the AI model
texts = df['content'].to_list()
print(f"\nReady to analyse {len(texts)} reviews")

### Step 4: Load the AI model and run it

We load the `distilbert` model from Hugging Face. This model:
- Was trained on thousands of English reviews
- Understands context — "not bad" = positive, "not good" = negative
- Returns a label (POSITIVE/NEGATIVE) and a confidence score (0-1)

First, let's test it on 3 simple examples to see how it works.

In [ ]:
# Load the pre-trained sentiment model
model_id = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"
sentiment_pipeline = pipeline("sentiment-analysis", model=model_id)
print("Model loaded successfully.")

In [ ]:
# Quick test with 3 examples — see how the model thinks
test_texts = [
    "Amazing satay, best I ever had!",
    "Terrible food, very disappointing",
    "It was okay, nothing special"
]

results = sentiment_pipeline(test_texts)
for text, result in zip(test_texts, results):
    print(f"  \"{text}\"")
    print(f"    → {result['label']} (confidence: {result['score']:.1%})\n")

In [ ]:
# Now run it on ALL 800 reviews and measure how long it takes
start = time.time()
predictions = sentiment_pipeline(texts)
end = time.time()

print(f"Processed {len(texts)} reviews in {int(end - start)} seconds")
print(f"\nFirst 5 predictions:")
for i in range(5):
    print(f"  Review: \"{texts[i][:60]}...\"")
    print(f"  Result: {predictions[i]['label']} ({predictions[i]['score']:.1%})\n")

### Step 5: How accurate is the model?

We compare the AI's predictions to our ground truth (the actual star ratings). This tells us:
- **Accuracy** — what % did it get right overall?
- **Confusion matrix** — where specifically did it make mistakes?
- **Precision & Recall** — is it better at detecting positive or negative reviews?

In [ ]:
# Compare predictions to ground truth
predict_pd = pd.DataFrame(predictions)
y_test = df['sentiment']          # actual sentiment (from ratings)
y_pred = predict_pd['label']      # AI-predicted sentiment (from text)

accuracy = accuracy_score(y_test, y_pred)
print(f"Overall Accuracy: {accuracy:.1%}")
print(f"  → The AI correctly classified {accuracy:.1%} of reviews just from the text\n")

# Detailed breakdown
print(classification_report(y_test, y_pred))

In [ ]:
# Confusion Matrix — visual breakdown of correct vs incorrect predictions
# Read it as: rows = actual sentiment, columns = predicted sentiment
labels = ['POSITIVE', 'NEGATIVE']
cm = confusion_matrix(y_test, y_pred, labels=labels)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp.plot()
plt.title('Sentiment Analysis — Confusion Matrix')
plt.show()

print("How to read this:")
print(f"  Top-left ({cm[0][0]}): Correctly identified as POSITIVE")
print(f"  Top-right ({cm[0][1]}): Was POSITIVE but AI said NEGATIVE (missed)")
print(f"  Bottom-left ({cm[1][0]}): Was NEGATIVE but AI said POSITIVE (missed)")
print(f"  Bottom-right ({cm[1][1]}): Correctly identified as NEGATIVE")

### Step 6: Vendor-level insights

This is the **real business value** — instead of just overall accuracy, we break down sentiment **per vendor**. This lets the admin see which stalls have the happiest customers and which ones need attention.

In [ ]:
# Add AI predictions back to the data
df['hf_predicted'] = predict_pd['label'].values
df['hf_confidence'] = predict_pd['score'].values

# Calculate sentiment stats per vendor
vendor_sentiment = df.groupby('vendor_name').agg(
    total_reviews=('review_id', 'count'),
    avg_rating=('rating', 'mean'),
    positive_pct=('hf_predicted', lambda x: (x == 'POSITIVE').mean() * 100),
    avg_confidence=('hf_confidence', 'mean'),
).sort_values('positive_pct', ascending=False).reset_index()

print("=== VENDOR SENTIMENT REPORT ===")
print("(This is what the Admin Dashboard shows)\n")
print(vendor_sentiment.to_string(index=False))

In [ ]:
# Visualise: Best and worst vendors side by side
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

top10 = vendor_sentiment.head(10)
axes[0].barh(top10['vendor_name'], top10['positive_pct'], color='#10b981', alpha=0.8)
axes[0].set_xlabel('Positive Sentiment %')
axes[0].set_title('Top 10 Vendors — Highest Customer Satisfaction')
axes[0].set_xlim(0, 100)

bottom10 = vendor_sentiment.tail(10)
axes[1].barh(bottom10['vendor_name'], bottom10['positive_pct'], color='#ef4444', alpha=0.8)
axes[1].set_xlabel('Positive Sentiment %')
axes[1].set_title('Bottom 10 Vendors — Need Improvement')
axes[1].set_xlim(0, 100)

plt.tight_layout()
plt.show()

---
# Part B: Sentiment Analysis with Ollama LLM

An alternative approach: instead of a specialised sentiment model, we use a **general-purpose large language model** (like ChatGPT, but running locally). We simply ask it: *"Is this review positive or negative?"*

**Trade-off**: More flexible (can handle nuanced language), but much slower since it processes one review at a time.

> **Note:** This section requires Ollama to be installed locally. If it's not available, it will skip gracefully.

### Step 1: Check if Ollama is available

In [ ]:
# Try to import ollama — it's okay if it's not installed
try:
    import ollama
    OLLAMA_AVAILABLE = True
    print("Ollama is available — will run LLM sentiment analysis")
except ImportError:
    OLLAMA_AVAILABLE = False
    print("Ollama not installed — skipping Part B")
    print("To install: pip install ollama (and run 'ollama serve' + 'ollama pull llama3')")

### Step 2: Use a smaller sample

LLMs process text much slower than specialised models (seconds per review vs milliseconds). We use a **sample of 50 reviews** to keep runtime reasonable.

In [ ]:
# Take a random sample of 50 reviews for LLM processing
df_sample = df.sample(n=50, random_state=42).reset_index(drop=True)
print(f"Sample: {len(df_sample)} reviews")
print(df_sample[['content', 'rating', 'sentiment']].head())

### Step 3: Run the LLM on each review

We send each review to the LLM with a prompt: *"Analyse this text and respond with ONLY ONE WORD: POSITIVE or NEGATIVE."* The LLM reads the review, understands the context, and responds.

In [ ]:
if OLLAMA_AVAILABLE:
    model_id = "llama3"

    # Quick test with one example
    test_prompt = "You are a sentiment analysis expert. Analyze the sentiment of this text:'Amazing satay, best at the pasar malam!' and respond with ONLY ONE WORD: 'POSITIVE' or 'NEGATIVE'."
    response = ollama.generate(model=model_id, prompt=test_prompt)
    print(f"Test: 'Amazing satay...' → {response['response'].strip()}")

    # Process all 50 reviews
    def llm_sa(row, model_id):
        sa_prompt = "You are a sentiment analysis expert. Analyze the sentiment of this text:'{}' and respond with ONLY ONE WORD: 'POSITIVE' or 'NEGATIVE'.".format(row['content'])
        response = ollama.generate(model=model_id, prompt=sa_prompt)
        return response['response'].upper().strip()

    start = time.time()
    df_sample['llama3_response'] = df_sample.apply(lambda row: llm_sa(row, model_id), axis=1)
    end = time.time()
    print(f"\nProcessed {len(df_sample)} reviews in {int(end - start)} seconds")
else:
    print("Skipping — Ollama not available")

### Step 4: Evaluate the LLM's accuracy

Same evaluation as Part A — compare the LLM's predictions to the actual star ratings.

In [ ]:
if OLLAMA_AVAILABLE and 'llama3_response' in df_sample.columns:
    accuracy_llm = accuracy_score(df_sample['sentiment'], df_sample['llama3_response'])
    print(f"LLM Accuracy: {accuracy_llm:.1%}")
    print(classification_report(df_sample['sentiment'], df_sample['llama3_response']))
else:
    print("Skipping — no LLM results available")

---
## Summary: Which approach is better?

| | Hugging Face (distilbert) | Ollama LLM (llama3) |
|---|---|---|
| **Speed** | ~2 seconds for 800 reviews | ~3 minutes for 50 reviews |
| **Accuracy** | High | High |
| **Setup** | `pip install transformers` | Requires Ollama server |
| **Best for** | Batch processing thousands of reviews | Nuanced analysis of individual reviews |

**For Bazaario**, we use Hugging Face in production (fast enough for real-time dashboards) and LLM as a fallback for edge cases that need deeper understanding.

### How this connects to the web app

The sentiment scores computed here are exported to the website and displayed in:
- **Vendor Analytics → Reviews tab**: The vendor sees their own review sentiment breakdown
- **Admin Dashboard → Reviews tab**: The admin sees all vendors ranked by customer satisfaction
- **Admin Dashboard → Vendors tab**: Sentiment ranking with progress bars per vendor